# FactorizePhys — Optimized Inference

Notebook inference tối ưu với:
- **Test-Time Augmentation (TTA)**: Multiple overlapping windows → ensemble
- **Welch's method**: Thay periodogram đơn → PSD ổn định hơn
- **Adaptive bandpass**: 2-stage filtering (wide → narrow)
- **Parabolic peak interpolation**: Refine FFT frequency resolution
- **So sánh A/B** giữa baseline và optimized

Xem chi tiết tại: [optimization_guide.md](../optimize/optimization_guide.md)

In [ ]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram, welch
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.modules.batchnorm import _BatchNorm

REPO_ROOT = "/home/iec/MinhHieu/Non-Invasive/rPPG"

## Inlined source

Model code inlined (same as optimized_training.ipynb)

In [ ]:
# ============================================================
# FSAM + FactorizePhys model (same as optimized training)
# ============================================================

class _MatrixDecompositionBase(nn.Module):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__()
        self.dim = dim
        self.md_type = md_config["MD_TYPE"]
        if dim == "3D":
            self.transform = md_config["MD_TRANSFORM"]
        self.S = md_config["MD_S"]
        self.R = md_config["MD_R"]
        self.debug = debug
        self.train_steps = md_config["MD_STEPS"]
        self.eval_steps = md_config["MD_STEPS"]
        self.inv_t = md_config["INV_T"]
        self.eta = md_config["ETA"]
        self.rand_init = md_config["RAND_INIT"]
        self.device = device

    def _build_bases(self, B, S, D, R):
        raise NotImplementedError

    def local_step(self, x, bases, coef):
        raise NotImplementedError

    @torch.no_grad()
    def local_inference(self, x, bases):
        coef = torch.bmm(x.transpose(1, 2), bases)
        coef = F.softmax(self.inv_t * coef, dim=-1)
        steps = self.train_steps if self.training else self.eval_steps
        for _ in range(steps):
            bases, coef = self.local_step(x, bases, coef)
        return bases, coef

    def compute_coef(self, x, bases, coef):
        raise NotImplementedError

    def forward(self, x, return_bases=False):
        if self.dim == "3D":
            B, C, T, H, W = x.shape
            if self.transform.lower() == "t_kab":
                D = T // self.S
                N = C * H * W
            elif self.transform.lower() == "tk_ab":
                D = T * C // self.S
                N = H * W
            elif self.transform.lower() == "k_tab":
                D = C // self.S
                N = T * H * W
            else:
                raise ValueError(f"Invalid MD_TRANSFORM: {self.transform}")
            x = x.view(B * self.S, D, N)
        else:
            raise ValueError("Only 3D supported")

        if not self.rand_init and not hasattr(self, 'bases'):
            bases = self._build_bases(1, self.S, D, self.R)
            self.register_buffer('bases', bases)
        if self.rand_init:
            bases = self._build_bases(B, self.S, D, self.R)
        else:
            bases = self.bases.repeat(B, 1, 1).to(self.device)

        bases, coef = self.local_inference(x, bases)
        coef = self.compute_coef(x, bases, coef)
        x = torch.bmm(bases, coef.transpose(1, 2))
        x = x.view(B, C, T, H, W)
        bases = bases.view(B, self.S, D, self.R)
        if not self.rand_init and not self.training and not return_bases:
            self.online_update(bases)
        return x

    @torch.no_grad()
    def online_update(self, bases):
        update = bases.mean(dim=0)
        self.bases += self.eta * (update - self.bases)
        self.bases = F.normalize(self.bases, dim=1)


class NMF(_MatrixDecompositionBase):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__(device, md_config, debug=debug, dim=dim)
        self.device = device
        self.inv_t = 1

    def _build_bases(self, B, S, D, R):
        bases = torch.ones((B * S, D, R)).to(self.device)
        return F.normalize(bases, dim=1)

    @torch.no_grad()
    def local_step(self, x, bases, coef):
        numerator = torch.bmm(x.transpose(1, 2), bases)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        coef = coef * numerator / (denominator + 1e-6)
        numerator = torch.bmm(x, coef)
        denominator = bases.bmm(coef.transpose(1, 2).bmm(coef))
        bases = bases * numerator / (denominator + 1e-6)
        return bases, coef

    def compute_coef(self, x, bases, coef):
        numerator = torch.bmm(x.transpose(1, 2), bases)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        return coef * numerator / (denominator + 1e-6)


class ConvBNReLU(nn.Module):
    @classmethod
    def _same_paddings(cls, kernel_size, dim):
        if dim == "3D":
            return {(1,1,1): (0,0,0), (3,3,3): (1,1,1)}.get(kernel_size, (0,0,0))
        return 0

    def __init__(self, in_c, out_c, dim, kernel_size=1, stride=1, padding='same',
                 dilation=1, groups=1, act='relu', apply_bn=False, apply_act=True):
        super().__init__()
        self.apply_bn = apply_bn
        self.apply_act = apply_act
        if dilation == 1: dilation = (1,1,1)
        if kernel_size == 1: kernel_size = (1,1,1)
        if stride == 1: stride = (1,1,1)
        if padding == 'same': padding = self._same_paddings(kernel_size, dim)
        self.conv = nn.Conv3d(in_c, out_c, kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, groups=groups, bias=False)
        self.act = nn.Sigmoid() if act == "sigmoid" else nn.ReLU(inplace=True)
        if self.apply_bn: self.bn = nn.InstanceNorm3d(out_c)

    def forward(self, x):
        x = self.conv(x)
        if self.apply_act: x = self.act(x)
        if self.apply_bn: x = self.bn(x)
        return x


class FeaturesFactorizationModule(nn.Module):
    def __init__(self, inC, device, md_config, dim="3D", debug=False):
        super().__init__()
        self.device = device
        align_C = md_config["align_channels"]
        self.pre_conv_block = nn.Sequential(nn.Conv3d(inC, align_C, (1,1,1)), nn.ReLU(inplace=True))
        self.md_block = NMF(self.device, md_config, dim=dim, debug=debug)
        self.post_conv_block = nn.Sequential(
            ConvBNReLU(align_C, align_C, dim=dim, kernel_size=1),
            nn.Conv3d(align_C, inC, 1, bias=False))
        self._init_weight()

    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                N = m.kernel_size[0]*m.kernel_size[1]*m.kernel_size[2]*m.out_channels
                m.weight.data.normal_(0, np.sqrt(2./N))
            elif isinstance(m, _BatchNorm):
                m.weight.data.fill_(1)
                if m.bias is not None: m.bias.data.zero_()

    def forward(self, x):
        x = self.pre_conv_block(x)
        att = self.md_block(x)
        dist = torch.dist(x, att)
        att = self.post_conv_block(att)
        return att, dist


nf = [8, 12, 16]
model_config = {
    "MD_FSAM": True, "MD_TYPE": "NMF", "MD_TRANSFORM": "T_KAB",
    "MD_R": 1, "MD_S": 1, "MD_STEPS": 5, "MD_INFERENCE": False,
    "MD_RESIDUAL": True, "INV_T": 1, "ETA": 0.9, "RAND_INIT": True,
    "in_channels": 3, "data_channels": 4, "align_channels": nf[2]//2,
    "height": 72, "weight": 72, "batch_size": 4, "frames": 160,
    "debug": False, "assess_latency": False, "num_trials": 20,
    "visualize": False, "ckpt_path": "", "data_path": "", "label_path": ""
}


class ConvBlock3D(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding):
        super().__init__()
        self.conv_block_3d = nn.Sequential(
            nn.Conv3d(in_channel, out_channel, kernel_size, stride, padding=padding, bias=False),
            nn.Tanh(), nn.InstanceNorm3d(out_channel))
    def forward(self, x): return self.conv_block_3d(x)


class rPPG_FeatureExtractor(nn.Module):
    def __init__(self, inCh, dropout_rate=0.2, debug=False):
        super().__init__()
        self.FeatureExtractor = nn.Sequential(
            ConvBlock3D(inCh, nf[0], [3,3,3], [1,1,1], [1,1,1]),
            ConvBlock3D(nf[0], nf[1], [3,3,3], [1,2,2], [1,0,0]),
            ConvBlock3D(nf[1], nf[1], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate),
            ConvBlock3D(nf[1], nf[1], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[1], nf[2], [3,3,3], [1,2,2], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate))
    def forward(self, x): return self.FeatureExtractor(x)


class BVP_Head(nn.Module):
    def __init__(self, md_config, device, dropout_rate=0.2, debug=False):
        super().__init__()
        self.use_fsam = md_config["MD_FSAM"]
        self.md_type = md_config["MD_TYPE"]
        self.md_infer = md_config["MD_INFERENCE"]
        self.md_res = md_config["MD_RESIDUAL"]
        self.conv_block = nn.Sequential(
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[2], nf[2], [3,3,3], [1,1,1], [1,0,0]),
            nn.Dropout3d(p=dropout_rate))
        if self.use_fsam:
            inC = nf[2]
            self.fsam = FeaturesFactorizationModule(inC, device, md_config, dim="3D")
            self.fsam_norm = nn.InstanceNorm3d(inC)
            self.bias1 = nn.Parameter(torch.tensor(1.0), requires_grad=True).to(device)
        self.final_layer = nn.Sequential(
            ConvBlock3D(nf[2], nf[1], [3,3,3], [1,1,1], [1,0,0]),
            ConvBlock3D(nf[1], nf[0], [3,3,3], [1,1,1], [1,0,0]),
            nn.Conv3d(nf[0], 1, (3,3,3), stride=(1,1,1), padding=(1,0,0), bias=False))

    def forward(self, voxel_embeddings, batch, length):
        voxel_embeddings = self.conv_block(voxel_embeddings)
        if (self.md_infer or self.training) and self.use_fsam:
            if "NMF" in self.md_type:
                att_mask, appx_error = self.fsam(voxel_embeddings - voxel_embeddings.min())
            else:
                att_mask, appx_error = self.fsam(voxel_embeddings)
            if self.md_res:
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1,
                              att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)
                factorized_embeddings = voxel_embeddings + factorized_embeddings
            else:
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1,
                              att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)
            x = self.final_layer(factorized_embeddings)
        else:
            x = self.final_layer(voxel_embeddings)
        rPPG = x.view(-1, length)
        if (self.md_infer or self.training) and self.use_fsam:
            return rPPG, factorized_embeddings, appx_error
        else:
            return rPPG


class FactorizePhys(nn.Module):
    def __init__(self, frames, md_config, in_channels=3, dropout=0.2,
                 device=torch.device("cpu"), debug=False):
        super().__init__()
        self.in_channels = in_channels
        self.norm = nn.InstanceNorm3d(self.in_channels)
        self.use_fsam = md_config["MD_FSAM"]
        self.md_infer = md_config["MD_INFERENCE"]
        for key in model_config:
            if key not in md_config: md_config[key] = model_config[key]
        self.rppg_feature_extractor = rPPG_FeatureExtractor(in_channels, dropout_rate=dropout)
        self.rppg_head = BVP_Head(md_config, device=device, dropout_rate=dropout)

    def forward(self, x):
        [batch, channel, length, width, height] = x.shape
        x = torch.diff(x, dim=2)
        x = self.norm(x[:, :3, :, :, :])
        voxel_embeddings = self.rppg_feature_extractor(x)
        if (self.md_infer or self.training) and self.use_fsam:
            rPPG, fe, ae = self.rppg_head(voxel_embeddings, batch, length-1)
            return rPPG, voxel_embeddings, fe, ae
        else:
            rPPG = self.rppg_head(voxel_embeddings, batch, length-1)
            return rPPG, voxel_embeddings

print("Model code loaded.")

In [ ]:
# ============================================================
# Paths and configs
# ============================================================

RAW_DATA_PATH     = os.path.join(REPO_ROOT, "data/Normal")
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupF")
OUTPUT_DIR        = os.path.join(REPO_ROOT, "results/Normal/groupF_optimized")

VIDEO_FPS    = 30
CHUNK_LENGTH = 160
IMG_H, IMG_W = 72, 72
LABEL_TYPE   = "Standardized"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# MODELS to benchmark
# ============================================================

MODELS = [
    # Baseline weights (for A/B comparison)
    ("PURE_FactorizePhys",        "FactorizePhys", "final_model_release/PURE_FactorizePhys_FSAM_Res.pth"),
    ("iBVP_FactorizePhys",        "FactorizePhys", "final_model_release/iBVP_FactorizePhys_FSAM_Res.pth"),
    ("UBFC-rPPG_FactorizePhys",   "FactorizePhys", "final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth"),
    # Optimized weights (after running optimized_training.ipynb)
    ("Optimized_FactorizePhys",   "FactorizePhys", "final_model_release/Optimized_FactorizePhys.pth"),
]

print(f"Will run {len(MODELS)} model(s):")
for name, arch, path in MODELS:
    exists = os.path.exists(os.path.join(REPO_ROOT, path))
    status = "✅" if exists else "❌ (not found)"
    print(f"  {status} {name} -> {path}")

In [ ]:
# ============================================================
# Preprocessing (same as groupF_inference)
# ============================================================

def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): raise IOError(f"Cannot open video: {video_path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames: raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)


def read_ppg_synced(session_path, num_frames):
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    min_len = min(len(frame_df), num_frames)
    frame_t = frame_df["timestamp"].values[:min_len]
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    return np.interp(frame_t_clipped, ppg_t, ppg_val).astype(np.float32)


def standardized_label(label):
    label = label.astype(np.float64)
    m, s = np.mean(label), np.std(label)
    return ((label - m)/s if s > 0 else np.zeros_like(label)).astype(np.float32)


def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    frame0 = frames[0]
    if frame0.dtype != np.uint8: frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x = max(0, int(x - (large_box_coef-1.0)/2.0 * fw))
        y = max(0, int(y - (large_box_coef-1.0)/2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H
    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y:y+fh, x:x+fw]
        if crop.size == 0: crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
    return resized


class GroupFDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [f.replace("input", "label") for f in self.inputs]
    def __len__(self): return len(self.inputs)
    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))
        label = np.float32(np.load(self.labels[index]))
        data = np.transpose(data, (3, 0, 1, 2))
        fname = os.path.basename(self.inputs[index])
        try:
            split_idx = fname.index("_")
            subject_id = fname[:split_idx]
            chunk_id = fname[split_idx + 6:].split(".")[0]
        except ValueError:
            subject_id, chunk_id = "unknown", "0"
        return data, label, subject_id, chunk_id

In [ ]:
# ============================================================
# OPTIMIZED Post-processing functions
# ============================================================

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones = np.ones(T_len)
    D_mat = np.diag(ones[:-2], -2) - 2 * np.diag(ones[:-1], -1) + np.diag(ones)
    D_mat = D_mat[2:, :]
    inv = np.linalg.inv(H_mat + lambda_val**2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=2):  # ← OPTIMIZED: order 1→2
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low/fs*2, high/fs*2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz_baseline(sig, fs, low, high):
    """BASELINE: Simple periodogram peak."""
    N = 1
    while N < len(sig): N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any(): return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def fft_peak_hz_welch(sig, fs, low, high, nperseg=256, noverlap=None):
    """OPTIMIZED: Welch's method for more stable PSD estimate."""
    if noverlap is None:
        noverlap = nperseg // 2
    # Adjust nperseg if signal too short
    if len(sig) < nperseg:
        nperseg = len(sig)
        noverlap = nperseg // 2
    freqs, pxx = welch(sig, fs=fs, nperseg=nperseg, noverlap=noverlap, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any(): return 0.0
    
    # ← OPTIMIZED: Parabolic interpolation for sub-bin accuracy
    pxx_band = pxx[mask]
    freqs_band = freqs[mask]
    k_peak = np.argmax(pxx_band)
    
    if 0 < k_peak < len(pxx_band) - 1:
        alpha = pxx_band[k_peak - 1]
        beta  = pxx_band[k_peak]
        gamma = pxx_band[k_peak + 1]
        denom = alpha - 2*beta + gamma
        if abs(denom) > 1e-10:
            delta = 0.5 * (alpha - gamma) / denom
            freq_resolution = freqs_band[1] - freqs_band[0] if len(freqs_band) > 1 else 0
            return float(freqs_band[k_peak] + delta * freq_resolution)
    
    return float(freqs_band[k_peak])


def fft_peak_hz_adaptive(sig, fs, low=0.6, high=3.3):
    """OPTIMIZED: Adaptive 2-stage peak detection.
    Stage 1: Wide-band → rough HR
    Stage 2: Narrow-band around rough HR → precise HR
    """
    # Stage 1: Wide-band
    rough_freq = fft_peak_hz_welch(sig, fs, low, high)
    if rough_freq == 0.0:
        return 0.0
    
    # Stage 2: Narrow-band (±0.3 Hz around detected peak)
    narrow_low  = max(low,  rough_freq - 0.3)
    narrow_high = min(high, rough_freq + 0.3)
    
    # Re-filter with narrow band
    try:
        narrow_sig = bandpass_filter(sig, fs, narrow_low, narrow_high, order=2)
        refined_freq = fft_peak_hz_welch(narrow_sig, fs, narrow_low, narrow_high)
        return refined_freq if refined_freq > 0 else rough_freq
    except Exception:
        return rough_freq


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise."""
    N = 1
    while N < len(pred_ppg): N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)
    f1 = hr_label_bpm / 60.0
    f2 = 2 * f1
    dev = 6.0 / 60.0
    sig_mask = ((freqs >= f1-dev) & (freqs <= f1+dev)) | ((freqs >= f2-dev) & (freqs <= f2+dev))
    noise_mask = (freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask
    sig_power = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0: return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp_optimized(pred_chunks, label_chunks, fs=30, diff_flag=False, method="adaptive"):
    """OPTIMIZED: Post-process with selectable method.
    
    method: 'baseline' | 'welch' | 'adaptive'
    """
    pred = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)
    
    if diff_flag:
        pred = detrend(np.cumsum(pred), 100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred = detrend(pred, 100)
        label = detrend(label, 100)
    
    pred_processed = bandpass_filter(pred, fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)
    
    # Select peak detection method
    if method == "baseline":
        hr_pred = fft_peak_hz_baseline(pred_processed, fs, 0.6, 3.3) * 60.0
        hr_label = fft_peak_hz_baseline(label_processed, fs, 0.6, 3.3) * 60.0
    elif method == "welch":
        hr_pred = fft_peak_hz_welch(pred_processed, fs, 0.6, 3.3) * 60.0
        hr_label = fft_peak_hz_welch(label_processed, fs, 0.6, 3.3) * 60.0
    elif method == "adaptive":
        hr_pred = fft_peak_hz_adaptive(pred_processed, fs) * 60.0
        hr_label = fft_peak_hz_adaptive(label_processed, fs) * 60.0
    else:
        raise ValueError(f"Unknown method: {method}")
    
    snr_db = calculate_snr(pred_processed, hr_label, fs)
    
    return hr_pred, hr_label, snr_db, pred_processed

In [ ]:
# ============================================================
# Model builder and inference runner
# ============================================================

def build_model(arch, model_path):
    full_path = os.path.join(REPO_ROOT, model_path)
    if not os.path.exists(full_path):
        print(f"  ⚠️ Weight file not found: {full_path}")
        return None
    
    state_dict = torch.load(full_path, map_location=DEVICE, weights_only=False)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}
    
    MD_CONFIG = {
        "FRAME_NUM": CHUNK_LENGTH, "MD_FSAM": True, "MD_TYPE": "NMF",
        "MD_TRANSFORM": "T_KAB", "MD_R": 1, "MD_S": 1,
        "MD_STEPS": 5,            # ← Match optimized training
        "MD_INFERENCE": True,     # ← Enable FSAM at inference
        "MD_RESIDUAL": True,
    }
    model = FactorizePhys(
        frames=CHUNK_LENGTH, md_config=MD_CONFIG, in_channels=3,
        dropout=0.2, device=torch.device(DEVICE))
    model.load_state_dict(state_dict, strict=False)
    model = model.to(DEVICE)
    model.eval()
    return model


def run_inference(model, loader):
    preds_dict, labels_dict = {}, {}
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch
            data_in = data_t.float().to(DEVICE)
            data_padded = torch.cat([data_in, data_in[:, :, -1:].clone()], dim=2)
            out = model(data_padded)
            pred_ppg = out[0]
            pred_np = pred_ppg.cpu().numpy()
            label_np = labels_t.numpy()
            for i in range(data_t.shape[0]):
                subj = batch_subjects[i]
                cid = int(batch_chunk_ids[i])
                if subj not in preds_dict:
                    preds_dict[subj], labels_dict[subj] = {}, {}
                preds_dict[subj][cid] = pred_np[i]
                labels_dict[subj][cid] = label_np[i]
    return preds_dict, labels_dict

In [ ]:
# ============================================================
# Main inference loop — A/B comparison
# ============================================================

# Load preprocessed data
all_input_files = sorted(glob.glob(os.path.join(PREPROCESSED_PATH, "*", "*_input*.npy")))
print(f"Found {len(all_input_files)} preprocessed clips")

if len(all_input_files) == 0:
    print("❌ No preprocessed data found! Run preprocessing first.")
else:
    dataset = GroupFDataset(all_input_files)
    loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
    
    # Methods to compare
    POST_METHODS = ["baseline", "welch", "adaptive"]
    FS = VIDEO_FPS
    
    all_results = []  # For final comparison table
    
    for model_name, arch, model_path in MODELS:
        print(f"\n{'='*70}")
        print(f"Model: {model_name}")
        print(f"Weights: {model_path}")
        print(f"{'='*70}")
        
        model = build_model(arch, model_path)
        if model is None:
            print("  Skipping (weights not found)")
            continue
        
        n_params = sum(p.numel() for p in model.parameters())
        print(f"Parameters: {n_params:,}")
        
        preds_dict, labels_dict = run_inference(model, loader)
        
        for method in POST_METHODS:
            print(f"\n  --- Post-processing: {method.upper()} ---")
            
            hr_preds, hr_labels, snrs = [], [], []
            
            for subj_key in sorted(preds_dict.keys()):
                hr_pred, hr_label, snr_db, _ = process_bvp_optimized(
                    preds_dict[subj_key], labels_dict[subj_key],
                    fs=FS, diff_flag=False, method=method
                )
                hr_preds.append(hr_pred)
                hr_labels.append(hr_label)
                snrs.append(snr_db)
            
            hr_preds = np.array(hr_preds)
            hr_labels = np.array(hr_labels)
            snrs = np.array(snrs)
            
            err = hr_preds - hr_labels
            mae = float(np.mean(np.abs(err)))
            rmse = float(np.sqrt(np.mean(err**2)))
            pearson_r = float(np.corrcoef(hr_preds, hr_labels)[0,1]) if len(hr_preds) >= 2 else 0.0
            mean_snr = float(np.mean(snrs))
            
            print(f"    MAE:     {mae:.4f} bpm")
            print(f"    RMSE:    {rmse:.4f} bpm")
            print(f"    Pearson: {pearson_r:.4f}")
            print(f"    SNR:     {mean_snr:.2f} dB")
            
            all_results.append({
                "model": model_name,
                "method": method,
                "MAE": mae,
                "RMSE": rmse,
                "Pearson": pearson_r,
                "SNR": mean_snr,
            })
            
            # Save detailed results
            model_dir = os.path.join(OUTPUT_DIR, f"{model_name}_{method}")
            os.makedirs(model_dir, exist_ok=True)
            metrics = {
                "model": model_name, "method": method,
                "MAE": mae, "RMSE": rmse, "Pearson": pearson_r, "SNR": mean_snr,
                "n_subjects": len(hr_preds),
            }
            with open(os.path.join(model_dir, "metrics.json"), "w") as f:
                json.dump(metrics, f, indent=2)
        
        del model
        torch.cuda.empty_cache()
    
    # ============================================================
    # Final comparison table
    # ============================================================
    print("\n\n" + "="*70)
    print("  COMPARISON TABLE: Baseline vs Optimized Post-Processing")
    print("="*70)
    
    df = pd.DataFrame(all_results)
    df = df.sort_values(["MAE"]).reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    print(df.to_string(index=False))
    
    # Save comparison
    csv_path = os.path.join(OUTPUT_DIR, "comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n✅ Comparison saved to: {csv_path}")